# Day 3

##  Question: How many machines to serve a 70B model?

This question is really asking: **how much GPU memory is required just to store the model weights**, and then **how many servers (“machines”) you need** given the number of GPUs per server.

For a first-pass estimate we usually ignore KV-cache and other runtime overheads (we’ll add those caveats after the math).

### Step 1: Compute weight memory (FP16)
A model with `P` parameters, stored in FP16, uses:

- `bytes_per_param = 2` bytes (FP16)

So weight size is:

- `weight_bytes = P * 2`

For `P = 70B = 70 x 10^9` parameters:

- `weight_bytes = 70e9 * 2 = 140e9 bytes`

Convert to GB / GiB:

- Decimal GB (common in marketing): `140e9 bytes ≈ 140 GB`
- Binary GiB (what GPUs often report internally): `140e9 / 2^30 ≈ 130.4 GiB`

### Step 2: Minimum number of 80GB GPUs to hold the weights
If a single GPU has about `80 GB` of usable memory, the minimum GPU count is:

- `gpus_needed = ceil(140 / 80) = 2`

So: **you need at least 2 × 80GB GPUs** to fit 70B parameters in FP16 *at minimum*.

> Important: in practice you rarely get to use 100% of advertised memory (CUDA/runtime overhead, memory fragmentation, etc.). This can push the requirement to **2 GPUs with a bit of margin**, or sometimes **3+ GPUs** depending on context length and serving setup.

### Step 3: Turn “GPUs needed” into “machines needed”
“Machines” depends on how many GPUs are in one server.

- If your machine has **1 GPU**: machines = 2
- If your machine has **8 GPUs (e.g., a typical DGX/HGX-style server)**: machines = 1, because you can place/shard the model across **multiple GPUs within the same server** (e.g., tensor parallelism).

So the corrected takeaway is:

- **Minimum GPUs** for FP16 weights: **2**
- **Minimum machines**: **depends on GPUs-per-machine**; commonly **1 machine** if it has multiple 80GB GPUs.

### Step 4 (caveat): KV cache + concurrency often dominate at inference
For real serving, GPU memory is not only weights.

- **KV cache** grows with `sequence_length` and `batch_size` (and also depends on model architecture and precision).
- **Concurrency** (how many requests you batch together) increases KV memory.

This means even if FP16 weights fit on 2 GPUs, you may still need **more GPUs** to support:

- long context windows
- higher throughput / higher concurrent requests
- larger batch sizes

In system design interviews, it’s good to explicitly state: “2 GPUs to fit weights; add extra for KV cache and target throughput.”


## Interview-ready answer (what to say)

If asked “How many machines to serve a 70B model?” I’d say something like:

1. **Weight memory (FP16):** `70B * 2 bytes = ~140 GB` of parameters.
2. **GPU requirement:** with `A100 ~80GB`, `ceil(140/80) = 2` GPUs are the **minimum** to fit the weights (using tensor parallel).
3. **Convert to machines using GPUs per server:**
   - `2 GPUs / machine` would be `ceil(2/2)=1 machine`.
   - `1 GPU / machine` would be `ceil(2/1)=2 machines`.
4. **Add serving headroom:** real serving also needs **KV cache** for attention and some overhead, so to hit latency/throughput targets you often allocate **extra GPUs** (and sometimes use quantization like FP8/INT8/4-bit).


## Practice: Latency vs Throughput trade-offs (LLM serving)

In interviews, “latency vs throughput” is usually about how you batch and parallelize work on GPUs.

- **Latency (p50/p95):** how fast a *single request* finishes (queueing wait + compute time).
- **Throughput:** how many requests/tokens you complete per unit time (e.g., requests/sec or tokens/sec).

### The key trade-off
If you increase batching and/or concurrency to improve GPU utilization:

- **Throughput goes up** because GPUs spend less time idle and you amortize overhead (kernel launches, attention setup, etc.).
- **Latency can go up** because requests may **wait** to be grouped into a batch (and because more load increases queueing).

### Main knobs you can mention
- **Batching strategy**
  - *Smaller batch size / more frequent dispatch:* lower waiting time -> lower latency, potentially lower throughput.
  - *Larger batch size / larger batching window:* higher throughput, but higher per-request latency.
- **Request scheduling**
  - Separate work into *prefill* (prompt processing) and *decode* (token-by-token generation).
  - Use priority policies (e.g., shorter prompts / interactive traffic first) to control tail latency.
- **Concurrency / autoscaling**
  - More concurrent requests increases utilization but also increases queue length.
  - Autoscale based on queue depth or GPU utilization to stay within latency SLOs.
- **Parallelism (multi-GPU / tensor parallel)**
  - Improves capacity (more model shards / more replicas), often helps throughput.
  - Can slightly increase per-request overhead (communication), so latency depends on network + sharding setup.
- **Decoding optimizations**
  - Techniques like speculative decoding, better caching, quantization, and optimized KV-cache management can improve *both* latency and throughput.

### How to answer in 30 seconds
“Latency vs throughput is governed by batching and queueing. Larger batches improve GPU utilization and throughput, but they introduce waiting time and can increase tail latency due to longer queues. In practice we pick a batching window and concurrency level that meets our p95/p99 latency SLO while keeping GPUs sufficiently utilized, and we often treat prefill and decode differently.”

### Practice scenario (try answering out loud)
You serve an LLM with the following constraints:

- Traffic is spiky (interactive users + some longer-form generations).
- Target **p95 latency <= 800 ms** for interactive prompts.
- During peak, GPU utilization drops if you dispatch too aggressively.

Questions:
1. What would you change first: batch size, batching window, or concurrency?
2. How would you treat short interactive prompts vs long prompts?
3. What metrics would you monitor to avoid “high throughput but bad tail latency”?
